# Experimento pareado: selección por F1 frente a F2 en LSTM32 dropout 0,2

Se comparan dos entrenamientos desde cero con los mismos datos, arquitectura, semilla, BCE, Adam, learning rate y regularización. Solo cambia el objetivo de selección: F2 frente a F1 para elegir checkpoint y umbral.

Se incluye además un control separado de umbral F1 sobre los pesos F2 ya congelados, sin reentrenar. El entrenamiento requiere GPU CUDA; 2025 permanece reservado.

## 1. Subir y verificar el paquete

Desde la raíz del repositorio genera el paquete corregido con `python3.13 scripts/empaquetar_lstm_f1.py --output runs/lstm_colab_f1_v2.zip`. Guarda el SHA-256 que imprime el empaquetador y sube **ese ZIP v2** al panel Archivos de Colab. El ZIP v1 contiene el runner anterior y no debe usarse. Selecciona GPU.


In [ ]:
from pathlib import Path
import hashlib, json, stat, subprocess, sys, zipfile
from google.colab import files

ROOT = Path('/content/f1-lstm-f1-v2')
PACKAGE = Path('/content/lstm_colab_f1_v2.zip')
EXPECTED_PACKAGE_SHA256 = '0023a176c7f3298b2df5a0c731495c6dfc073513ef51b2e2dd26eb71e8956a3c'
if ROOT.exists(): raise FileExistsError('Usa una sesión limpia: /content/f1-lstm-f1-v2 ya existe.')
if not PACKAGE.is_file():
    uploaded = files.upload()
    if len(uploaded) != 1: raise ValueError('Sube exactamente el ZIP v2 del par LSTM F1/F2.')
    name, content = next(iter(uploaded.items()))
    if Path(name).name != 'lstm_colab_f1_v2.zip': raise ValueError('Se requiere lstm_colab_f1_v2.zip; el ZIP v1 está desactualizado.')
    PACKAGE.write_bytes(content)
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('SHA-256 del paquete:', package_hash)
if package_hash != EXPECTED_PACKAGE_SHA256: raise ValueError('El hash no coincide con el ZIP v2 corregido.')
with zipfile.ZipFile(PACKAGE) as z:
    members=z.infolist(); names=set()
    for m in members:
        p=Path(m.filename)
        if p.is_absolute() or '..' in p.parts or '\\' in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest=json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'lstm_colab_f1_inputs_v1' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    expected=set(manifest['files_sha256']) | {'colab_package_manifest.json'}
    if names != expected: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT/name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; sin datos de prueba.')


## 2. Instalar dependencias y confirmar GPU

La comparación se cancela antes de entrenar si Colab no expone CUDA. No cambies a CPU ni cambies el tipo de entorno entre los dos modelos.

In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/content/f1-lstm-f1-v2')
if not (ROOT / 'requirements-colab-experiments.txt').is_file():
    raise RuntimeError('No encuentro el paquete extraído en /content/f1-lstm-f1-v2. Ejecuta primero la sección 1 desde una sesión Colab activa.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-colab-experiments.txt')], check=True)
import torch
cuda_available = torch.cuda.is_available()
print('PyTorch:', torch.__version__, '| build CUDA:', torch.version.cuda, '| cuda.is_available:', cuda_available, '| dispositivos visibles:', torch.cuda.device_count())
if not cuda_available:
    if torch.version.cuda is None:
        detail = 'Este runtime tiene un build CPU de PyTorch.'
    else:
        detail = 'PyTorch tiene soporte CUDA, pero no ve un dispositivo GPU conectado.'
    raise RuntimeError(detail + ' Elimina el runtime actual, reconecta una sesión Colab GPU y vuelve a ejecutar; el notebook no instalará ni sustituirá torch.')
print('GPU:', torch.cuda.get_device_name(0))

## 3. Entrenar la pareja

La referencia selecciona checkpoint y umbral por F2; la candidata lo hace por F1. Ambas usan early stopping en validación a umbral fijo 0,5, paciencia seis y desempate por BCE menor. Se reportan F1 y F2 junto a precisión, recall, AP, BCE, confusión y cobertura de eventos. La validación es de desarrollo, no prueba independiente.

In [ ]:
import json, subprocess, sys
from pathlib import Path
from google.colab import files
ROOT = Path('/content/f1-lstm-f1-v2')
if not (ROOT / 'scripts/entrenar_comparacion_lstm_f1.py').is_file():
    raise RuntimeError('No encuentro el runner corregido en el paquete v2. Ejecuta primero la sección 1 desde una sesión Colab activa.')
RUN = ROOT/'runs/lstm32_f1_objective_v1_seed42'
if RUN.exists() or RUN.with_suffix('.zip').exists(): raise FileExistsError('La salida ya existe. Conserva los artefactos y usa una sesión/salida nueva.')
result = subprocess.run(
    [sys.executable, 'scripts/entrenar_comparacion_lstm_f1.py', '--reference-config', 'configs/experimento_lstm32_f1_ref_v1.json', '--candidate-config', 'configs/experimento_lstm32_f1_candidate_v1.json', '--output', str(RUN.relative_to(ROOT))],
    cwd=ROOT,
    check=False,
)
if result.returncode != 0:
    status_path = RUN/'status.json'
    if status_path.is_file():
        print('Estado del par:')
        print(json.dumps(json.loads(status_path.read_text()), indent=2))
    raise RuntimeError(f'La corrida falló con código {result.returncode}; revisa el estado anterior antes de reintentar.')
summary=json.loads((RUN/'pair_summary.json').read_text())
print(json.dumps(summary, indent=2))
files.download(str(RUN.with_suffix('.zip')))


## 4. Devolver resultados al proyecto

Descarga `lstm32_f1_objective_v1_seed42.zip` y colócalo intacto en `runs/`. La evaluación local CPU verificará hashes y paridad antes de comparar ambos modelos.